## 1-3 뉴스 키워드 추출

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 기본 LLM 제공자: "groq" 또는 "openai" — get_llm() 호출 시 provider 를 생략하면 이 값을 사용
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [3]:
llm = get_llm(temperature=0.7, seed=42, provider="groq")  # 제공자를 직접 지정 (키/모델명은 공통 설정 셀의 get_llm 사용)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{news}"),
    ("ai", "키워드: {keywords}")
])

examples = [
    {
        "news": (
            "국내 연구진이 사람의 말을 알아듣고 집안일을 돕는 가정용 AI 로봇을 개발했다. "
            "이 AI 로봇은 음성 명령만으로 청소와 설거지를 할 수 있으며, 사용자의 생활 패턴을 학습해 스스로 일정을 조정한다. "
            "연구진은 내년 상반기 AI 로봇의 시범 판매를 시작하고, 가격을 낮춘 보급형 로봇도 함께 선보일 계획이라고 밝혔다."
        ),
        "keywords": "AI, 로봇, 음성 명령, 시범 판매",
    },
    {
        "news": (
            "올여름 폭염이 이어지면서 전국의 전력 사용량이 역대 최고치를 기록했다. "
            "기상청은 폭염이 다음 주까지 계속될 것으로 예상하며, 노약자는 한낮 야외 활동을 자제해 달라고 당부했다. "
            "정부는 전력 수급 안정을 위해 공공기관의 냉방 온도를 조정하고 절전 캠페인을 확대하기로 했다."
        ),
        "keywords": "폭염, 전력, 기상청, 전력 수급, 절전",
    },
    {
        "news": (
            "한국은행이 기준금리를 0.25%포인트 인하하면서 대출 금리도 함께 내려갈 전망이다. "
            "전문가들은 이번 기준금리 인하가 위축된 소비와 투자를 살리는 데 도움이 될 것으로 내다봤다. "
            "다만 대출이 늘어나면서 부동산 가격이 다시 오를 수 있다는 우려도 나오고 있다."
        ),
        "keywords": "한국은행, 기준금리, 인하, 대출, 부동산",
    },
]

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# 최종 프롬프트
final_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 뉴스 기사에서 핵심 키워드를 추출하는 전문가입니다. "
     "기사에서 자주 등장하고 중요한 키워드 5개를 중요한 순서대로 쉼표로 구분해 답하세요. "
     "설명 없이 키워드만 출력하세요."),
    few_shot_prompt,          
    ("human", "{news}"),
])

chain = final_prompt | llm | StrOutputParser()



In [10]:
test_news_1 = (
    "교육부가 내년부터 전국 초등학교에 코딩 교육 시간을 두 배로 늘린다고 발표했다. "
    "새 교육과정에서는 학생들이 블록 코딩으로 시작해 간단한 게임과 앱을 직접 만들어보는 수업이 진행된다. "
    "교육부는 코딩 교육을 담당할 교사 연수를 확대하고, 학교마다 노트북 등 실습 장비도 지원할 계획이다. "
    "일부 학부모들은 코딩 교육 확대를 반기면서도 사교육 부담이 커질 수 있다고 걱정했다."
)




result = chain.invoke({"news": test_news_1})
print(result)

교육부, 코딩 교육, 초등학교, 교사 연수, 노트북
